# SL ladder · 01 · Grid to district-week exposure

Turns the cached ERA5 and CHIRPS grids into the three variables the ladder consumes -
`t2m_mean_c`, `precip_sum_mm`, `rh_mean_percent` - as an area-weighted district-week table.

Two things here are easy to get silently wrong, so both are checked rather than assumed:
the **grid index mapping** (the weight files and the `.npz` arrays use different origins) and
**CHIRPS nodata** (a plain weighted mean lets one ocean cell poison a whole district).

## 1 · Setup

In [1]:
from pathlib import Path
import numpy as np, pandas as pd

REPO = Path.cwd().parent if Path.cwd().name.startswith("notebooks") else Path.cwd()
DQ   = REPO / "data_quarantine"
ERA  = DQ / "wp5_exposure" / "era5_0p25_window"
CHI  = DQ / "wp5_exposure" / "chirps_window"
OUTD = DQ / "sl_ladder"
OUTD.mkdir(parents=True, exist_ok=True)

YEARS = range(2018, 2026)
we = pd.read_csv(DQ / "wp5_exposure/wp5_buildB_weights_era5_0p25_srilanka_2020.csv")
wc = pd.read_csv(DQ / "wp5_exposure/wp5_buildB_weights_chirps_p05_srilanka_2020.csv")
print("weights:", we.shape, wc.shape)

weights: (260, 10) (3044, 9)


## 2 · Line the grids up

The weight files index columns from -180 deg; the ERA5 `.npz` indexes them from 0 deg E. The offset is
720 cells at 0.25 deg. CHIRPS is simpler - its window is exactly the weight-file bounding box.

Both mappings are asserted into range, and then a physical check: the first district's temperatures
must look like Sri Lankan temperatures. Index arithmetic that lands in the Bay of Bengal still
produces numbers; it does not produce 26 C.

In [2]:
z0 = np.load(ERA / "era5_0p25_t2m_2018.npz")
r0, r1, c0, c1 = z0["window"]
nr, nc = z0["data"].shape[1], z0["data"].shape[2]

we["ai"] = we.cell_row - int(r0)
we["aj"] = we.cell_col - (int(c0) + 720)
assert we.ai.between(0, nr - 1).all() and we.aj.between(0, nc - 1).all(), "ERA5 index out of range"

p0 = np.load(CHI / "chirps_p05_srilanka_2018.npz")["precip"]
# CHIRPS: cell_row is RASTER order (north-down); the staged window is stored in the netCDF's
# LATITUDE-ASCENDING order (south-up). Subtracting the min flips the island north-to-south, and the
# bbox assert below cannot see it because the flipped index range is identical. Flip explicitly.
wc["aj"] = wc.cell_col - int(wc.cell_col.min())
wc["ai"] = (p0.shape[1] - 1) - (wc.cell_row - int(wc.cell_row.min()))
assert (wc.ai.max(), wc.aj.max()) == (p0.shape[1] - 1, p0.shape[2] - 1), "CHIRPS bbox mismatch"

probe = z0["data"][:, we.ai.iloc[0], we.aj.iloc[0]]
print(f"probe cell t2m: {probe.min()-273.15:.1f} to {probe.max()-273.15:.1f} C")
assert 15 < probe.min() - 273.15 and probe.max() - 273.15 < 45, "not a tropical land cell"

probe cell t2m: 22.2 to 33.0 C


In [3]:
# ORIENTATION GATE — the bbox assert above passes under a north-south flip, so check GEOGRAPHY.
# Sri Lanka's wet zone is the southwest (Ratnapura, Kegalle); the north (Jaffna, Mannar) is dry.
_p = np.where(p0 < -100, np.nan, p0).sum(axis=0)          # 2018 annual total per cell
_ann = {gid: float(np.nansum(_p[d.ai.values, d.aj.values] * d.w_area.values) / d.w_area.sum())
        for gid, d in wc.groupby("geometry_id")}
_name = wc.drop_duplicates("geometry_id").set_index("geometry_id").rdhs_name
_ann = pd.Series(_ann).rename(_name).sort_values(ascending=False)
print("wettest 3:", ", ".join("%s %.0f mm" % (k, v) for k, v in _ann.head(3).items()))
print("driest  3:", ", ".join("%s %.0f mm" % (k, v) for k, v in _ann.tail(3).items()))
assert _ann["Ratnapura"] > _ann["Jaffna"] * 1.5, (
    "CHIRPS rows are upside-down: the southwest wet zone is not wetter than the north")
print("\norientation OK — the wet zone is in the southwest, where it belongs")

# and a direct cross-check against WP5's independently-built twin, which exists in the quarantine
_w5 = pd.read_csv(DQ / "wp5_exposure/wp5_precip_exposure_twin_srilanka_v1.csv", parse_dates=["week_start"])
print("cross-check against wp5_01's Build A' available:", len(_w5), "rows")


wettest 3: Kalutara 3821 mm, Colombo 3413 mm, Galle 3411 mm
driest  3: Mannar 1206 mm, Jaffna 1052 mm, Killinochchi 996 mm

orientation OK — the wet zone is in the southwest, where it belongs
cross-check against wp5_01's Build A' available: 10842 rows


## 3 · Relative humidity

ERA5 ships dewpoint, not RH. Magnus formula, applied hourly and then averaged - averaging the inputs
first and converting once would bias the result, because the relation is nonlinear.

In [4]:
def rh_from_t_d(t_k, d_k):
    """Magnus RH% from 2m temperature and dewpoint, both Kelvin."""
    t, d = t_k - 273.15, d_k - 273.15
    a, b = 17.625, 243.04
    return 100.0 * np.exp(a * d / (b + d) - a * t / (b + t))

def era_daily(year):
    t = np.load(ERA / f"era5_0p25_t2m_{year}.npz")
    d = np.load(ERA / f"era5_0p25_d2m_{year}.npz")
    dates = pd.to_datetime(t["dates"]); n = len(dates)
    tk = t["data"].reshape(n, 24, nr, nc)
    dk = d["data"].reshape(n, 24, nr, nc)
    return dates, tk.mean(axis=1) - 273.15, rh_from_t_d(tk, dk).mean(axis=1)

## 4 · Weighted aggregation, NaN-aware

CHIRPS masks water, and 34% of the cells in this window are ocean. A plain `values @ weights` turns
any district touching the coast into `NaN`. Instead the weights are renormalised over the cells that
are valid, per day.

One district - **LK52K (Kalmunai)**, a thin east-coast strip - has *no* valid CHIRPS cell at all, so it
falls back to the nearest valid cell. That is a real limitation of 0.05 deg rainfall over narrow coastal
units, and it is printed rather than hidden.

In [5]:
def weighted(grid, w, wcol="w_area"):
    """grid (ndays, nr, nc) -> DataFrame(day x district), renormalising over valid cells."""
    vals = grid[:, w.ai.values, w.aj.values]
    ok = np.isfinite(vals)
    any_ok = ok.any(axis=0)
    gi, gj = w.ai.values, w.aj.values
    out, cov = {}, {}
    for gid, idx in w.groupby("geometry_id").groups.items():
        pos = w.index.get_indexer(idx)
        ww = w.loc[idx, wcol].values
        v, o = vals[:, pos], ok[:, pos]
        den = o @ ww
        cov[gid] = float((ww * o.any(axis=0)).sum() / ww.sum())
        s = np.where(den > 0, (np.where(o, v, 0.0) @ ww) / np.where(den > 0, den, 1.0), np.nan)
        if not np.isfinite(s).any():
            ci = (w.loc[idx, "ai"] * ww).sum() / ww.sum()
            cj = (w.loc[idx, "aj"] * ww).sum() / ww.sum()
            d2 = (gi[any_ok] - ci) ** 2 + (gj[any_ok] - cj) ** 2
            src = np.flatnonzero(any_ok)[int(np.argmin(d2))]
            s = vals[:, src]
            print(f"    fallback: {gid} has no valid cell -> nearest at ({gi[src]},{gj[src]})")
        out[gid] = s
    return pd.DataFrame(out), cov

## 5 · Build the daily table

In [6]:
def to_long(df, dates, name):
    return (df.assign(date=dates).melt(id_vars="date", var_name="geometry_id", value_name=name)
              .set_index(["date", "geometry_id"])[name])

parts = {"t2m_mean_c": [], "rh_mean_percent": [], "precip_sum_mm": []}
for y in YEARS:
    if not (ERA / f"era5_0p25_t2m_{y}.npz").exists():
        print(f"  skip {y}: no ERA5"); continue
    dates, t2m_c, rh = era_daily(y)
    cz = np.load(CHI / f"chirps_p05_srilanka_{y}.npz")
    cdates = pd.to_datetime(cz["dates"])
    precip = np.where(cz["precip"] < -100, np.nan, cz["precip"])   # nodata is -9999
    dt, _ = weighted(t2m_c, we); dr, _ = weighted(rh, we); dp, coverage = weighted(precip, wc)
    parts["t2m_mean_c"].append(to_long(dt, dates, "t2m_mean_c"))
    parts["rh_mean_percent"].append(to_long(dr, dates, "rh_mean_percent"))
    parts["precip_sum_mm"].append(to_long(dp, cdates, "precip_sum_mm"))
    print(f"  {y}: {len(dates)}d")

daily = pd.concat({k: pd.concat(v, axis=0) for k, v in parts.items()}, axis=1).reset_index()
VARS = ["t2m_mean_c", "rh_mean_percent", "precip_sum_mm"]
assert daily.geometry_id.nunique() == 26 and daily[VARS].notna().all().all()
print("\ndaily", daily.shape)
print(daily[VARS].describe().round(2).to_string())

  2018: 365d


  2019: 365d


  2020: 366d


  2021: 365d


  2022: 365d


  2023: 365d


  2024: 366d


  2025: 365d

daily (75972, 5)
       t2m_mean_c  rh_mean_percent  precip_sum_mm
count    75972.00         75972.00       75972.00
mean        26.75            81.01           6.13
std          2.02             7.37          10.68
min         17.57            49.77           0.00
25%         25.57            76.41           0.00
50%         26.79            82.00           1.13
75%         28.20            86.66           8.19
max         32.16            99.12         156.96


## 6 · CHIRPS coverage, per district

Three districts sit below 25% area coverage. Their rainfall is carried by one or two cells, which is
worth remembering whenever a district-level rainfall effect looks unusual.

In [7]:
cov = pd.Series(coverage).sort_values()
print("lowest coverage districts:"); print(cov.head(6).round(3).to_string())
print(f"\nbelow 25% coverage: {(cov < 0.25).sum()} of {len(cov)}")
cov.to_frame("chirps_area_coverage").to_csv(OUTD / "sl_chirps_coverage.csv")

lowest coverage districts:
LK62    0.996
LK41    0.997
LK42    0.998
LK53    0.999
LK12    0.999
LK11    0.999

below 25% coverage: 0 of 26


## 7 · Weekly, Monday-start

Weeks are keyed by their Monday. Partial weeks at either end are dropped rather than scaled, so every
row is a full seven days.

In [8]:
daily["week_start"] = daily["date"] - pd.to_timedelta(daily["date"].dt.weekday, unit="D")
wk = (daily.groupby(["geometry_id", "week_start"])
      .agg(t2m_mean_c=("t2m_mean_c", "mean"), rh_mean_percent=("rh_mean_percent", "mean"),
           precip_sum_mm=("precip_sum_mm", "sum"), ndays=("date", "size")).reset_index())
wk = wk[wk.ndays == 7].drop(columns="ndays")
print("weekly", wk.shape, "|", wk.week_start.min().date(), "->", wk.week_start.max().date())
print(wk[VARS].describe().round(2).to_string())
wk.to_csv(OUTD / "sl_exposure_weekly_0p25_area.csv", index=False)
# GATE — weekly rainfall must agree with wp5_01's independently-built Build A' (fractional area).
# Same CHIRPS window, same weights, a different notebook: an exact match is the strongest available
# check that this table is not silently mis-indexed.
_chk = wk.merge(_w5[["geometry_id", "week_start", "precip_sum_mm_a_frac"]],
                on=["geometry_id", "week_start"], how="inner")
_d = (_chk.precip_sum_mm - _chk.precip_sum_mm_a_frac).abs()
print("\nvs wp5_01 Build A': %d rows compared | max|diff| %.3e mm | corr %.6f"
      % (len(_chk), _d.max(), _chk[["precip_sum_mm", "precip_sum_mm_a_frac"]].corr().iloc[0, 1]))
assert _d.max() < 1e-3, "weekly rainfall disagrees with wp5_01 — check the CHIRPS row mapping"

print("\nwrote", (OUTD / "sl_exposure_weekly_0p25_area.csv").relative_to(REPO))

weekly (10842, 5) | 2018-01-01 -> 2025-12-22
       t2m_mean_c  rh_mean_percent  precip_sum_mm
count    10842.00         10842.00       10842.00
mean        26.76            81.00          42.94
std          1.95             6.68          50.02
min         19.66            56.19           0.00
25%         25.64            76.89           5.85
50%         26.78            82.27          24.93
75%         28.16            86.21          63.22
max         31.54            94.03         375.50



vs wp5_01 Build A': 10842 rows compared | max|diff| 4.167e-06 mm | corr 1.000000

wrote data_quarantine/sl_ladder/sl_exposure_weekly_0p25_area.csv
